# Project starter

This is the plumbing, so you spend your two days on your client and not on `ColumnTransformer`
errors. It runs end to end as it stands. It is **deliberately mediocre**, because two things are
left for you: cleaning the data, and everything to do with your client.

Nothing here is specific to any brief. All seven teams get the same file.

**Rules of use.** Change anything. Delete anything. If you disagree with a choice made here, say
so in the defence and change it; that is worth marks, not lost marks.

**One thing you must keep:** the function in section 6 must run on a file it has never seen. On
Friday you will be handed a fresh CSV and given fifteen minutes to produce your client's output
from it. If your work only exists as cells you ran by hand in order, you will not manage it.

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

SEED = 42
raw = pd.read_csv("C:\\Users\\Kiran Shinde\\OneDrive\\Desktop\\MLAP\\car_listings_for_project.csv")
print(raw.shape)
raw.head()

(4255, 15)


,listing_id,brand,model_year,mileage_km,engine_litres,fuel_type,transmission,body_type,owners,service_history,accident_reported,city,days_on_market,price_eur,sold_fast
0,L100394,Renault,2011,321312.0,1.9,petrol,manual,sedan,4.0,partial,0,Kazan,62,2400.0,0
1,L100940,Kia,2015,116734.0,1.7,petrol,automatic,hatchback,3.0,NaN,1,Novosibirsk,42,5070.0,0
2,L103383,Ford,2015,157618.0,0.9,petrol,automatic,suv,3.0,NaN,1,Yekaterinburg,9,4580.0,1
3,L100930,Kia,2017,60355.0,2.0,petrol,manual,hatchback,3.0,none,0,Novosibirsk,52,6570.0,0
4,L103083,Volkswagen,2020,81292.0,2.1,hybrid,automatic,suv,3.0,full,0,Yekaterinburg,100,17460.0,0


In [15]:
DATA_PATH = "C:\\Users\\Kiran Shinde\\OneDrive\\Desktop\\MLAP\\car_listings_for_project.csv"   # ask the instructor if this path fails

df = pd.read_csv(DATA_PATH)

print("Rows and columns:", df.shape)
df.head(10)
df.info()

Rows and columns: (4255, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4255 entries, 0 to 4254
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   listing_id         4255 non-null   object 
 1   brand              4255 non-null   object 
 2   model_year         4255 non-null   int64  
 3   mileage_km         4073 non-null   float64
 4   engine_litres      4038 non-null   float64
 5   fuel_type          4255 non-null   object 
 6   transmission       4255 non-null   object 
 7   body_type          4175 non-null   object 
 8   owners             4143 non-null   float64
 9   service_history    3953 non-null   object 
 10  accident_reported  4255 non-null   int64  
 11  city               4255 non-null   object 
 12  days_on_market     4255 non-null   int64  
 13  price_eur          4255 non-null   float64
 14  sold_fast          4255 non-null   int64  
dtypes: float64(4), int64(4), object(7)
memory u

## 1. Look at the file before you model it

The cell below does not fix anything. It points at things worth looking at. Some of what it
shows is a real problem, some is normal. Deciding which is which is your first job, and
"what was wrong with the raw file and what did you do about it" is a question you will be asked
on Friday.

In [16]:
print("=== missing values, % of rows ===")
print((raw.isna().mean() * 100).round(1)[lambda s: s > 0].to_string())

print("\n=== numeric ranges: does anything look impossible? ===")
print(raw.describe().T[["min", "25%", "50%", "75%", "max"]].to_string())

print("\n=== text columns: how many distinct values, and are they written consistently? ===")
TEXT = ["brand", "fuel_type", "transmission", "body_type", "service_history", "city"]
for c in TEXT:
    print(f"{c:20s} {raw[c].nunique():3d} distinct  {sorted(raw[c].dropna().unique())[:6]}")

print(f"\n=== identical rows (ignoring listing_id): "
      f"{raw.duplicated(subset=[c for c in raw.columns if c != 'listing_id']).sum()} ===")

=== missing values, % of rows ===
mileage_km         4.3
engine_litres      5.1
body_type          1.9
owners             2.6
service_history    7.1

=== numeric ranges: does anything look impossible? ===
                      min      25%       50%       75%        max
model_year         1900.0   2014.0    2016.0    2019.0     2025.0
mileage_km           -1.0  68566.0  114136.0  179519.0  4012300.0
engine_litres         0.9      1.5       1.8       2.1        3.8
owners                1.0      1.0       2.0       3.0        5.0
accident_reported     0.0      0.0       0.0       0.0        1.0
days_on_market        1.0      7.0      35.0      72.0      251.0
price_eur           900.0   5105.0    8090.0   12930.0    61500.0
sold_fast             0.0      0.0       0.0       1.0        1.0

=== text columns: how many distinct values, and are they written consistently? ===
brand                 16 distinct  [' BMW ', ' Ford ', ' Hyundai ', ' Kia ', ' Renault ', ' Skoda ']
fuel_type       

### Your cleaning goes here

Decide what to do about whatever you found above. Options for any given problem: fix it, set it
to missing and let the imputer handle it, or drop the row. Each is defensible; guessing is not.

Two hints that are not answers:

* A value can be wrong in a way that is *recoverable*. If a number is off by exactly a factor of
  ten, you know both that it is wrong and what it should be.
* Setting something to missing is not a cop-out. `NaN` is handled by the pipeline below. A wrong
  number is not.

You must also create `age_years`, because the year a car was made is not what drives its price;
how old it is now, is. The current year is 2026.

In [5]:
def clean(df):

    d = df.copy()

    # Drop duplicates
    d = d.drop_duplicates(subset=[c for c in d.columns if c != "listing_id"]).reset_index(drop=True)


# fix recoverable factor-of-10
    d.loc[d["mileage_km"] > 1_000_000, "mileage_km"]/=10

# impossible value - NaN
    d.loc[d["mileage_km"] < 0, "mileage_km"] = np.nan
    d.loc[d["model_year"] < 1990,"model_year"] = np.nan

    for c in ["brand", "fuel_type", "transmission",
            "body_type", "service_history", "city"]:
            #add .str.lower() to remove inconsistency
            d[c] = d[c].astype(str).str.strip().str.lower()

    d["age_years"] = 2026 - d["model_year"]

    return d

data = clean(raw)
print(f"{len(raw)} rows in, {len(data)} rows out")
print(f"Missing after clean:\n{data.isna().sum()[data.isna().sum() > 0]}")

4255 rows in, 4200 rows out
Missing after clean:
model_year        14
mileage_km       205
engine_litres    214
owners           109
age_years         14
dtype: int64


## 2. Choose the columns

`days_on_market` is **not** in the list below and must not be added. It is only known after a car
has sold, so it would not exist when your client needs an answer. Using it is called leakage and
it costs marks in every brief.

`sold_fast` is a thing to *predict*, not a thing to predict *from*. Only two briefs need it.

In [18]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED= 42

NUM = ["age_years", "mileage_km", "engine_litres", "owners", "accident_reported"]
CAT = ["brand", "fuel_type", "transmission", "body_type", "city", "service_history"]

def prep(num=NUM, cat=CAT):
    """Fills missing values, scales numbers, turns text into columns of 0s and 1s."""
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())]), num),
        ("cat", Pipeline([("ignore", SimpleImputer(strategy="constant", fill_value="unknown")),
                        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ])


print(" shape:", df.shape)  


 shape: (4255, 15)


## 3. Split, once, before you do anything else

Everything you try gets compared using `train`. The `test` half is touched **once**, at the end,
to produce the number you report. If you compare models by their test score, that score is no
longer honest, and you will be asked directly on Friday how many models you tried and when you
first looked at the test number.

Use `cross_val_score` on the training half to compare options. That is what it is for.

In [19]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

X = data[NUM + CAT]
y = np.log(data["price_eur"])
price = data["price_eur"]

Xtr, Xte, ytr, yte, ptr, pte = train_test_split(
    X, y, price, test_size=0.25, random_state=SEED
)
print(f"train {len(Xtr)}   test {len(Xte)}")

# Alignment checks — work for any split size (Friday's file may differ)
assert len(Xtr) == len(ytr) == len(ptr)
assert len(Xte) == len(yte) == len(pte)
assert np.allclose(np.exp(ytr), ptr.values), "ytr/ptr misaligned"
assert np.allclose(np.exp(yte), pte.values), "yte/pte misaligned"

ptr_ids = data.loc[ptr.index, "listing_id"]
pte_ids = data.loc[pte.index, "listing_id"]
overlap = set(ptr_ids) & set(pte_ids)
assert len(overlap) == 0, f"LEAK: {len(overlap)} listing_ids shared"

print(f"Xtr {Xtr.shape}   Xte {Xte.shape}")
print("All alignment checks passed — no leakage.")

train 3150   test 1050
Xtr (3150, 11)   Xte (1050, 11)
All alignment checks passed — no leakage.


## 4. The baseline you have to beat

The simplest thing that could serve a client: the median price of cars of the same brand and body
type. If your model does not beat this, you have nothing to sell. **Every brief requires you to
report this number.**

In [20]:
def baseline(X_train, y_train, X_test):
    """Median log-price of cars with the same (brand, body_type).
    Falls back to global median for unseen combinations."""
    train = X_train.copy()
    train["y"] = y_train.values

    grp = train.groupby(["brand", "body_type"])["y"].median()
    global_med = train["y"].median()

    keys = list(zip(X_test["brand"], X_test["body_type"]))
    preds = np.array([grp.get(k, global_med) for k in keys])
    return np.exp(preds)   # back to euros

base_pred = baseline(Xtr, ytr, Xte)
print(f"baseline MAE: {mean_absolute_error(pte, base_pred):,.0f} EUR")

    # raise NotImplementedError

# print(f"baseline MAE: {mean_absolute_error(pte, baseline(Xte)):,.0f} EUR")

baseline MAE: 4,836 EUR


## 5. A model

Linear regression on the log of the price. Try others by all means, but compare them with
`cross_val_score` on the **training** half, not on the test half.

Worth knowing before you spend an hour on it: a fancier model is not automatically better here,
and reporting honestly that something complicated performed worse is worth full marks.

In [21]:
base_mae    = mean_absolute_error(pte, base_pred)
model_mae   = mean_absolute_error(pte, pred_eur)
improvement = 100 * (base_mae - model_mae) / base_mae

print(f"Baseline MAE : {base_mae:>8,.0f} EUR")
print(f"Model MAE    : {model_mae:>8,.0f} EUR")
print(f"Improvement  : {improvement:>8.1f}% better than baseline")

NameError: name 'pred_eur' is not defined

## 6. The part that must survive Friday

Wrap everything into one function that goes from a raw file to your client's answer. Not a price
prediction: **your client's answer.** A bid, a lending value, a retail-or-auction decision, a
ranking, a list of flags.

On Friday you get a CSV in the same format with the price column removed, and fifteen minutes.
Test this now by pointing it at the training file: if it works on that, it will work on Friday.

In [ ]:
from sklearn.model_selection import KFold

# Honest residuals: fit on 4/5 of the training half, look at errors on 1/5
residuals = []
for tr, va in KFold(5, shuffle=True, random_state=SEED).split(Xtr):
    m = Pipeline([("p", prep()), ("m", LinearRegression())]).fit(Xtr.iloc[tr], ytr.iloc[tr])
    residuals.extend(ytr.iloc[va].values - m.predict(Xtr.iloc[va]))
residuals = np.array(residuals)

# Viktor's two costs, and the percentile they imply
COST_OVERPAY = 900
COST_LOSE    = 150
P_STAR = COST_LOSE / (COST_OVERPAY + COST_LOSE)     # = 150/1050 = 0.1429

print(f"Derived optimal percentile p* = {P_STAR:.4f}  ({P_STAR:.1%})")
print(f"offset at p*                  = {np.quantile(residuals, P_STAR):+.4f}")

In [ ]:
pred_log_te = model.predict(Xte)
actual_eur  = pte.values

rows = []
for p in np.round(np.arange(0.02, 0.51, 0.02), 3):
    offset  = np.quantile(residuals, p)
    bid_eur = np.exp(pred_log_te + offset)

    overpaid = bid_eur > actual_eur        # paid more than the car was worth
    lost     = ~overpaid                   # someone else won the car

    n_over = int(overpaid.sum())
    n_lose = int(lost.sum())

    mean_over = (bid_eur[overpaid] - actual_eur[overpaid]).mean() if n_over else 0.0
    mean_lost = (actual_eur[lost] - bid_eur[lost]).mean()        if n_lose else 0.0

    rows.append({
        "percentile":           p,
        "n_overpay":            n_over,
        "n_lose":               n_lose,
        "mean_overpaid_eur":    round(mean_over, 0),
        "mean_margin_lost_eur": round(mean_lost, 0),
        "total_loss_eur":       COST_OVERPAY * n_over + COST_LOSE * n_lose,
    })

sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

best = sweep.loc[sweep["total_loss_eur"].idxmin()]
print(f"\nDerived p*        : {P_STAR:.3f}")
print(f"Sweep best p      : {best['percentile']:.2f}")
print(f"Agreement         : {'YES' if abs(best['percentile'] - P_STAR) <= 0.05 else 'NO'}")
print(f"Mean overpaid     : {best['mean_overpaid_eur']:,.0f} EUR per won car")
print(f"Mean margin lost  : {best['mean_margin_lost_eur']:,.0f} EUR per lost car")
print(f"Total loss        : {best['total_loss_eur']:,.0f} EUR over {len(Xte)} cars")

In [ ]:
print("\n--- Why theory and sweep disagree ---")
print(f"Theory p*  = C_lose / (C_overpay + C_lose) = {P_STAR:.3f}")
print(f"Sweep p    = {best['percentile']:.2f}")
print()
print("Theory assumes model residuals = true price distribution.")
print("Sweep measures actual overpay/lose counts on held-out data.")
print("Our model is well-calibrated (R2≈0.96), so it overpays less often")
print("than the formula predicts. The sweep's answer is what actually")
print("happens in euros, so we adopt it as Viktor's bid percentile.")

In [ ]:
def advise(df, percentile=P_STAR):
    """Raw CSV in → Viktor's bid per car, in euros.
    Uses the derived percentile of the model's residual distribution
    to set the bid below the point estimate."""
    d = clean(df)
    pred_log = model.predict(d[NUM + CAT])
    offset   = np.quantile(residuals, percentile)
    bid_eur  = np.exp(pred_log + offset)
    return pd.DataFrame({
        "listing_id": d["listing_id"],
        "bid_eur":    bid_eur.round(0),
    })


advise(raw.head(20))

## 7. Trust check: 10 marks, and most teams will forget it

Your client will send you a car unlike anything you trained on: a brand you have never seen, an
odometer reading of 2,000,000, a form with half the fields blank. Right now your system answers
all of those with a confident number and no warning.

The skeleton below flags rows that do not look like the training data. **It is not finished and
one of its rules does not work.** Run the check in the next cell before you trust it.

In [ ]:


train_ranges = {
    "mileage_km":    (Xtr["mileage_km"].min(),    Xtr["mileage_km"].max()),
    "engine_litres": (Xtr["engine_litres"].min(), Xtr["engine_litres"].max()),
    "age_years":     (Xtr["age_years"].min(),     Xtr["age_years"].max()),
    "owners":        (Xtr["owners"].min(),        Xtr["owners"].max()),
}
train_brands = set(Xtr["brand"].dropna().unique())
train_fuels  = set(Xtr["fuel_type"].dropna().unique())
train_cities = set(Xtr["city"].dropna().unique())


def why_not_trusted(df):
    """One list of reasons per row. Empty list = row looks like training data."""
    reasons = []
    for _, row in df.iterrows():
        r = []

        # 1. Numeric outside training range
        for col, (lo, hi) in train_ranges.items():
            v = row.get(col, np.nan)
            if pd.notna(v) and (v < lo or v > hi):
                r.append(f"{col}={v} outside [{lo:.0f}, {hi:.0f}]")

        # 2. Unseen category
        if pd.notna(row["brand"])     and row["brand"]     not in train_brands:
            r.append(f"unseen brand: {row['brand']}")
        if pd.notna(row["fuel_type"]) and row["fuel_type"] not in train_fuels:
            r.append(f"unseen fuel: {row['fuel_type']}")
        if pd.notna(row["city"])      and row["city"]      not in train_cities:
            r.append(f"unseen city: {row['city']}")

        # 3. Too many blanks
        n_blank = sum(pd.isna(row[c]) for c in NUM + CAT)
        if n_blank >= 4:
            r.append(f"{n_blank} fields blank")

        reasons.append(r)
    return reasons

In [ ]:
# Does the detector actually catch rows the model gets wrong?
point_eur = np.exp(model.predict(Xte))
err       = np.abs(point_eur - pte.values) / pte.values
flagged   = np.array([len(r) > 0 for r in why_not_trusted(Xte)])

print(f"Flagged {flagged.mean():.1%} of the test set "
      f"({flagged.sum()} of {len(flagged)} rows)")

if flagged.any():
    print(f"Median error, flagged rows  : {np.median(err[flagged]):.1%}")
else:
    print("Median error, flagged rows  : — (nothing flagged)")

if (~flagged).any():
    print(f"Median error, unflagged rows: {np.median(err[~flagged]):.1%}")
else:
    print("Median error, unflagged rows: — (everything flagged)")

print("\nExample flagged rows:")
shown = 0
for i, r in enumerate(why_not_trusted(Xte)):
    if r:
        print(f"  row {i}: {r}")
        shown += 1
        if shown >= 3:
            break

In [ ]:
def advise(df, percentile=P_STAR):
    """Viktor's bid per car, with the trust check applied.
    Policy: no bid on untrusted rows — hand to a human."""
    d = clean(df)
    pred_log = model.predict(d[NUM + CAT])
    bid_eur  = np.exp(pred_log + np.quantile(residuals, percentile))

    reasons = why_not_trusted(d)
    trusted = [len(r) == 0 for r in reasons]

    out = pd.DataFrame({
        "listing_id":      d["listing_id"],
        "bid_eur":         bid_eur.round(0),
        "trusted":         trusted,
        "why_not_trusted": ["; ".join(r) if r else "" for r in reasons],
    })
    out.loc[~out["trusted"], "bid_eur"] = np.nan
    return out


advise(raw.head(20))

### Now check whether your detector is real

A detector that flags rows which turn out to be **no worse than average** is decoration. Test it:
compare the error on flagged rows against unflagged rows. If flagged rows are not worse, either
the rule is wrong or it is flagging the wrong thing. Find out which, fix it, and say what you
found. That is the difference between 2 marks and 3 here.

In [ ]:

train_ranges = {
    "mileage_km":    (Xtr["mileage_km"].min(),    Xtr["mileage_km"].max()),
    "engine_litres": (Xtr["engine_litres"].min(), Xtr["engine_litres"].max()),
    "age_years":     (Xtr["age_years"].min(),     Xtr["age_years"].max()),
    "owners":        (Xtr["owners"].min(),        Xtr["owners"].max()),
}
train_brands = set(Xtr["brand"].dropna().unique())
train_fuels  = set(Xtr["fuel_type"].dropna().unique())
train_cities = set(Xtr["city"].dropna().unique())


def why_not_trusted(df):
    """One list of reasons per row. Empty list = row looks like training data."""
    reasons = []
    for _, row in df.iterrows():
        r = []

        for col, (lo, hi) in train_ranges.items():
            v = row.get(col, np.nan)
            if pd.notna(v) and (v < lo or v > hi):
                r.append(f"{col}={v} outside [{lo:.0f}, {hi:.0f}]")

        if pd.notna(row["brand"])     and row["brand"]     not in train_brands:
            r.append(f"unseen brand: {row['brand']}")
        if pd.notna(row["fuel_type"]) and row["fuel_type"] not in train_fuels:
            r.append(f"unseen fuel: {row['fuel_type']}")
        if pd.notna(row["city"])      and row["city"]      not in train_cities:
            r.append(f"unseen city: {row['city']}")

        n_blank = sum(pd.isna(row[c]) for c in NUM + CAT)
        if n_blank >= 4:
            r.append(f"{n_blank} fields blank")

        reasons.append(r)
    return reasons

**Then decide what your system does about a flagged row.** Flagging alone is half the mark.
Refuse to answer, widen your margin, hand it to a human, put it at the bottom of the queue:
whichever fits your client. Say which, and why.

## 8. Known failures: 10 marks, and they must be specific

"The model is not perfect" is worth nothing. A usable failure statement names four things: the
**segment**, the **direction**, the **size**, and **what it means for your client**.

The cell below is a way of looking, not an answer. Slice by whatever you think might matter.

In [ ]:
# =============================================================================
# Section 8 — Known failures (stronger slices)
# =============================================================================
point_eur = np.exp(model.predict(Xte))
signed    = (point_eur - pte.values) / pte.values
abs_err   = np.abs(signed)

look = pd.DataFrame({
    "signed_err": signed, "abs_err": abs_err,
    "price":   pte.values,
    "age":     Xte["age_years"].values,
    "mileage": Xte["mileage_km"].values,
    "engine":  Xte["engine_litres"].values,
    "brand":   Xte["brand"].values,
    "body":    Xte["body_type"].values,
    "fuel":    Xte["fuel_type"].values,
})

print(f"Overall median abs error: {np.median(abs_err):.1%}\n")

# 1. Every brand × body_type with n >= 5 — worst 3 by |signed|
by_combo = (look.groupby(["brand", "body"])
            .agg(n=("abs_err", "size"),
                 med_abs=("abs_err", "median"),
                 med_sig=("signed_err", "median"))
            .query("n >= 5")
            .sort_values("med_sig", key=lambda s: s.abs(), ascending=False))
print("Worst 5 (brand, body_type) combos by |bias|:")
print(by_combo.head(5).round(3).to_string())
print()

# 2. Small engines (<= 1.2L) — often mis-specified
seg = look[look["engine"] <= 1.2]
print(f"Small engines (<= 1.2L):")
print(f"  n             = {len(seg)}")
if len(seg):
    print(f"  median |err|  = {seg['abs_err'].median():.1%}")
    print(f"  median signed = {seg['signed_err'].median()*100:+.1f}%")
    print(f"  euro bias     = {seg['signed_err'].median() * seg['price'].median():+,.0f} EUR\n")

# 3. Large engines (>= 2.5L)
seg = look[look["engine"] >= 2.5]
print(f"Large engines (>= 2.5L):")
print(f"  n             = {len(seg)}")
if len(seg):
    print(f"  median |err|  = {seg['abs_err'].median():.1%}")
    print(f"  median signed = {seg['signed_err'].median()*100:+.1f}%")
    print(f"  euro bias     = {seg['signed_err'].median() * seg['price'].median():+,.0f} EUR\n")

# 4. Age buckets — find the worst decade
look["age_bin"] = pd.cut(look["age"], bins=[0, 5, 10, 15, 20, 100],
                        labels=["0-5", "5-10", "10-15", "15-20", "20+"])
by_age = look.groupby("age_bin", observed=True).agg(
    n=("abs_err", "size"),
    med_abs=("abs_err", "median"),
    med_sig=("signed_err", "median"),
).round(3)
print("By age bucket:")
print(by_age.to_string())

## 9. Before Friday

- [ ] `clean()` and `advise()` run on a file they have never seen, in one go
- [ ] your baseline number is written down
- [ ] your headline is in euros, cars, or percent of listings, not R2
- [ ] the test half was scored once, and you can say how many models you tried
- [ ] your trust check flags rows that really are worse, and something happens when it fires
- [ ] two specific failure modes, with direction and size
- [ ] one question ready for the team you audit
- [ ] both of you can answer for the whole project, not half each